In [ ]:
import json
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# =========================================================
# CONFIG
# =========================================================
BASE_DIR = Path.cwd()


json_files = {
    "Baseline": BASE_DIR / "Baseline" / "xray_results.json",
    "Experiment_1": BASE_DIR / "Exp1" / "xray_results.json",
    "Experiment_2": BASE_DIR / "Exp2" / "xray_results.json",
    "Experiment_3": BASE_DIR / "Exp3" / "xray_results.json",
}

# Metrics where LOWER is better
LOWER_BETTER = ["MSE"]

# Metrics where HIGHER is better
HIGHER_BETTER = ["SSIM", "PSNR"]


# =========================================================
# LOAD JSONS
# =========================================================

all_results = []

for exp_name, file_path in json_files.items():

    with open(file_path, "r") as f:
        data = json.load(f)

    # Average metrics across all noise levels / keys
    mse_avg = []
    ssim_avg = []
    psnr_avg = []

    for level, metrics in data.items():
        mse_avg.append(metrics["MSE"])
        ssim_avg.append(metrics["SSIM"])
        psnr_avg.append(metrics["PSNR"])

    result = {
        "Experiment": exp_name,
        "MSE": sum(mse_avg) / len(mse_avg),
        "SSIM": sum(ssim_avg) / len(ssim_avg),
        "PSNR": sum(psnr_avg) / len(psnr_avg),
    }

    all_results.append(result)

# =========================================================
# CREATE COMPARISON TABLE
# =========================================================

df = pd.DataFrame(all_results)

print("\n================ RAW METRICS ================\n")
print(df)

# =========================================================
# NORMALIZATION
# Each metric gets equal weight
# =========================================================

normalized_df = df.copy()

# Handle metrics where LOWER is better
for metric in LOWER_BETTER:
    values = df[[metric]].values

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)

    # invert because lower is better
    normalized_df[metric + "_score"] = 1 - scaled

# Handle metrics where HIGHER is better
for metric in HIGHER_BETTER:
    values = df[[metric]].values

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(values)

    normalized_df[metric + "_score"] = scaled

# =========================================================
# FINAL EQUAL-WEIGHT SCORE
# =========================================================

score_columns = [
    "MSE_score",
    "SSIM_score",
    "PSNR_score",
]

normalized_df["Final_Score"] = normalized_df[score_columns].mean(axis=1)

# Sort best to worst
final_df = normalized_df.sort_values(
    by="Final_Score",
    ascending=False
).reset_index(drop=True)

print("\n================ FINAL COMPARISON ================\n")
print(
    final_df[
        [
            "Experiment",
            "MSE",
            "SSIM",
            "PSNR",
            "Final_Score",
        ]
    ]
)

# =========================================================
# BEST EXPERIMENT
# =========================================================

best_experiment = final_df.iloc[0]

print("\n=================================================")
print(f"BEST EXPERIMENT: {best_experiment['Experiment']}")
print("=================================================")



================ RAW METRICS ================

     Experiment       MSE      SSIM       PSNR
0      Baseline  0.005641  0.815130  22.591468
1  Experiment_1  0.006976  0.770848  21.588735
2  Experiment_2  0.004887  0.836126  23.208972
3  Experiment_3  0.004419  0.841443  23.783629

================ FINAL COMPARISON ================

     Experiment       MSE      SSIM       PSNR   Final_Score
0  Experiment_3  0.004419  0.841443  23.783629  1.000000e+00
1  Experiment_2  0.004887  0.836126  23.208972  8.266125e-01
2      Baseline  0.005641  0.815130  22.591468  5.354158e-01
3  Experiment_1  0.006976  0.770848  21.588735  7.401487e-17

BEST EXPERIMENT: Experiment_3
